# 05 · Joins

Joins combine two DataFrames by matching key columns. We'll learn every join type on a tiny example,
then use them on the shop data.

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
emp = spark.createDataFrame(
    [(1, "Ann", 10), (2, "Ben", 20), (3, "Cat", 30), (4, "Dev", None)],
    "emp_id int, name string, dept_id int")
dept = spark.createDataFrame(
    [(10, "Sales"), (20, "Engineering"), (40, "Legal")],
    "dept_id int, dept_name string")
emp.show(); dept.show()

In [ ]:
for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]:
    print(f"how='{how}'")
    emp.join(dept, on="dept_id", how=how).orderBy("emp_id").show()

| `how=` | Keeps |
|---|---|
| `inner` (default) | only rows with a match on both sides |
| `left` | every left row; right columns are null when no match |
| `right` | every right row |
| `full` | everything from both sides |
| `left_semi` | left rows that **have** a match — only left columns (a filter, like SQL `EXISTS`) |
| `left_anti` | left rows that have **no** match (like `NOT EXISTS`) — great for "customers who never…" |
| `cross` | every combination (n × m rows!) — use `crossJoin`, rarely what you want |

Notice: **null keys never match** — Dev (dept_id null) only appears in left/full/anti joins.

## The three ways to write `on=`

1. `on="dept_id"` or `on=["a", "b"]` — column name(s) that exist in **both**; the key appears **once** in the output. simplest
2. `on=emp.dept_id == dept.dept_id` — an expression; needed when names differ or for non-equality joins.
   The output then has **two** `dept_id` columns, which causes "ambiguous column" errors later.
3. Use **aliases** to keep things clear:

In [ ]:
joined = emp.join(dept, emp.dept_id == dept.dept_id)
joined.show()                                           # two dept_id columns
joined.select(emp.dept_id, "name", "dept_name").show()  # pick one explicitly

e, d = emp.alias("e"), dept.alias("d")
e.join(d, F.col("e.dept_id") == F.col("d.dept_id"), "left").select("e.name", "d.dept_name").show()

### Duplicate keys multiply rows
If a key appears 3 times on the left and 2 times on the right, you get 6 rows. Always know whether your
keys are unique! A quick check: `df.groupBy("key").count().filter("count > 1").show()`.

In [ ]:
dept_dupes = dept.union(dept)   # every department twice
print(emp.join(dept, "dept_id").count(), "→", emp.join(dept_dupes, "dept_id").count())

## Joining the shop data

In [ ]:
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA}/products.csv", header=True, inferSchema=True)
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)

enriched = (orders
            .join(products, "product_id")
            .join(customers, "customer_id")
            .withColumn("revenue", F.col("quantity") * F.col("price")))
enriched.select("order_id", "name", "country", "product_name", "category", "quantity", "price", "revenue").show(5)

## Broadcast joins
Normally to join, Spark **shuffles** both tables so matching keys end up on the same executor — expensive.
If one side is small (like `products`), Spark can instead **broadcast** — send a full copy of it to every
executor — and skip the shuffle entirely. Spark does this automatically for tables under 10 MB
(`spark.sql.autoBroadcastJoinThreshold`), and you can force it with `F.broadcast(...)`.
Look for `BroadcastHashJoin` in the plan:

In [ ]:
orders.join(F.broadcast(products), "product_id").explain()

### Exercise 1
Create `rev_by_category`: for **completed** orders only, total revenue (quantity × price) per category. Columns **`category`, `revenue`** (rounded to 2 dp), sorted by revenue descending.

In [ ]:
rev_by_category = None  # TODO

In [ ]:
# Check — run me after your answer
got = {r.category: r.revenue for r in rev_by_category.collect()}
exp = {'Books': 346914.81, 'Clothing': 174227.66, 'Electronics': 270553.15, 'Home': 234515.31, 'Toys': 374985.12}
assert got.keys() == exp.keys() and all(abs(got[k] - exp[k]) < 0.05 for k in exp), got
assert [r.category for r in rev_by_category.collect()] == sorted(exp, key=exp.get, reverse=True), "sort descending"
print("Correct!")

### Exercise 2
Create `never_ordered`: customers who have **never** placed an order (any status).

In [ ]:
never_ordered = None  # TODO

In [ ]:
# Check — run me after your answer
assert set(never_ordered.columns) == set(customers.columns), "should only have customer columns"
assert never_ordered.count() == 20
print("Correct!")

### Exercise 3
Create `top_countries`: the **3** countries (ignore null) with the highest completed-order revenue. Columns **`country`, `revenue`**, highest first.

In [ ]:
top_countries = None  # TODO

In [ ]:
# Check — run me after your answer
assert [r.country for r in top_countries.collect()] == ['Brazil', 'Japan', 'US']
print("Correct!")

### Exercise 4
Store in `n_book_buyers` the number of **distinct customers** with at least one **completed** order in the **Books** category.

In [ ]:
n_book_buyers = None  # TODO

In [ ]:
# Check — run me after your answer
assert n_book_buyers == 179, n_book_buyers
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/05_joins.ipynb`](../solutions/05_joins.ipynb).